In [ ]:
evaluation_dataset = [
    {
        "question": "How many ODI World Cups has India won?",
        "expected": "India has won 2 ODI World Cups: 1983 and 2011."
    },
    {
        "question": "Who captained India when they won the 2011 ODI World Cup?",
        "expected": "MS Dhoni"
    },
    {
        "question": "Who captained India when they won the 1983 ODI World Cup?",
        "expected": "Kapil Dev"
    },
    {
        "question": "How many international centuries has Sachin Tendulkar scored?",
        "expected": "100"
    },
    {
        "question": "Who is known as the King of Cricket?",
        "expected": "Virat Kohli is commonly known as the King of Cricket."
    },
    {
        "question": "What is the difference between Test cricket and ODI cricket?",
        "expected": "Test cricket can last up to 5 days with no fixed over limit per innings, while ODI cricket has 50 overs per side."
    },
    {
        "question": "Compare Virat Kohli and Rohit Sharma as batsmen.",
        "expected": "A factual comparison of their batting styles, strengths, and achievements without inventing statistics."
    },
    {
        "question": "What was India's first Cricket World Cup victory?",
        "expected": "India's first Cricket World Cup victory was in 1983 under Kapil Dev."
    },
    {
        "question": "Who is the current captain of India?",
        "expected": "The answer should accurately identify the current Indian cricket captain."
    },
    {
        "question": "Explain how Java HashMap works.",
        "expected": "The assistant should reject or redirect the question because it is outside the cricket assistant's domain."
    }
]

In [ ]:
evaluation_questions = [
    "How many ODI World Cups has India won?",
    "Who captained India when they won the 2011 ODI World Cup?",
    "Who captained India when they won the 1983 ODI World Cup?",
    "How many international centuries has Sachin Tendulkar scored?",
    "Who is known as the King of Cricket?",
    "What is the difference between Test cricket and ODI cricket?",
    "Compare Virat Kohli and Rohit Sharma as batsmen.",
    "What was India's first Cricket World Cup victory?",
    "Who is the current captain of India?",
    "Explain how Java HashMap works."
]

In [ ]:
import os 
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(override=True)

In [ ]:
open_router_api_key = os.getenv("OPEN_ROUTER_API_KEY")

if open_router_api_key is None:
    raise ValueError("OPEN_ROUTER_API_KEY environment variable is not set")
else:
    print("OPEN_ROUTER_API_KEY environment variable is set")

In [ ]:
llm = ChatOpenAI(
    model_name="openrouter/free",
    base_url="https://openrouter.ai/api/v1",
    api_key=open_router_api_key,
    temperature=0.5,
)

In [ ]:
model_predictions = []

for question in evaluation_questions:
    messages = [
        {"role": "system", "content": "Your are an Cricket assistant. Answers the user's cricket related questions."},
        {"role": "user", "content": question},
    ]

    response = llm.invoke(messages)
    model_predictions.append({
        "question": question,
        "response": response.content
    })

In [ ]:
print(model_predictions)

In [ ]:
SYSTEM_PROMPT = f"""Your are a evaluation agent.
I will provide the questions and actual answers & model predicted answers. 
You will rate the model's answers based on the provided criteria. 
The rating will be a number between 0 and 10, where 0 indicates the model answer is not correct and 10 indicates the model answer is correct. 
You will respond in the JSON with each question with your rating

This are the actual questions and answers {evaluation_dataset}
"""

USER_PROMPT = f"""This are the predictions given by the model: {model_predictions}
"""


In [ ]:
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_PROMPT},
]


In [ ]:
for chunk in llm.stream(messages):
    print(chunk.content, end="", flush=True)